# Checkpoint 2 — Mock Contest 2 Solutions

Reference solutions for all 6 problems. Each is a complete stdin/stdout program; the cells below are display-only mirrors of the solver files in the `assets/` folder.

## Question 1

**Notice:** sort the kids' smallest accepted sizes and the snack sizes. Walk through the snacks from smallest to largest while `kid` points at the least demanding kid still waiting. If the snack is big enough for that kid, that kid is happy and the next kid waits; otherwise the snack is too small for every waiting kid (they all need at least as much), so skip it. Giving the smallest workable snack to the least demanding kid never wastes a bigger snack that a harder kid could use. After two sorts the sweep moves each pointer forward at most once per item, so `100000` kids and snacks take about `N log N` steps.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
kid_count = int(tokens[0])
snack_count = int(tokens[1])
needs = []
i = 0
while i < kid_count:
    needs.append(int(tokens[2 + i]))
    i = i + 1
snacks = []
i = 0
while i < snack_count:
    snacks.append(int(tokens[2 + kid_count + i]))
    i = i + 1
needs.sort()
snacks.sort()

happy = 0
kid = 0
snack = 0
while kid < kid_count and snack < snack_count:
    if snacks[snack] >= needs[kid]:
        happy = happy + 1
        kid = kid + 1
    snack = snack + 1
print(str(happy))


## Question 2

**Notice:** keep the two amounts `water_a` and `water_b` and apply each command in order. `FILL` sets a jug to its capacity and `EMPTY` sets it to `0`. A pour moves `min(from jug's water, room left in the other jug)`, which is exactly where it stops: the first jug runs dry or the second becomes full. Each command is a constant amount of work, so `100000` commands are one quick pass, and Python's whole numbers handle capacities up to `10^9`.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
cap_a = int(tokens[0])
cap_b = int(tokens[1])
n = int(tokens[2])
water_a = 0
water_b = 0
position = 3
step = 0
while step < n:
    command = tokens[position]
    if command == "FILL":
        if tokens[position + 1] == "A":
            water_a = cap_a
        else:
            water_b = cap_b
        position = position + 2
    elif command == "EMPTY":
        if tokens[position + 1] == "A":
            water_a = 0
        else:
            water_b = 0
        position = position + 2
    else:
        if tokens[position + 1] == "A":
            moved = min(water_a, cap_b - water_b)
            water_a = water_a - moved
            water_b = water_b + moved
        else:
            moved = min(water_b, cap_a - water_a)
            water_b = water_b - moved
            water_a = water_a + moved
        position = position + 3
    step = step + 1
print(str(water_a) + " " + str(water_b))


## Question 3

**Notice:** build `prefix`, where `prefix[d]` is the total saved in the first `d` days. Because every amount is positive, the totals only grow, so for each goal binary-search for the first `d` from `1` to `N` with `prefix[d] >= goal`. When the search ends at `N + 1`, even the last day's total is short, so print `-1`. Building the array is one pass, and each goal takes about `17` halvings instead of rescanning `100000` days.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
q = int(tokens[1])
prefix = [0]
i = 0
while i < n:
    prefix.append(prefix[i] + int(tokens[2 + i]))
    i = i + 1

i = 0
while i < q:
    goal = int(tokens[2 + n + i])
    lo = 1
    hi = n + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if prefix[mid] >= goal:
            hi = mid
        else:
            lo = mid + 1
    if lo == n + 1:
        print("-1")
    else:
        print(str(lo))
    i = i + 1


## Question 4

**Notice:** sort the stations and keep a pointer to the first station not yet passed. While the destination is out of reach (`current + tank < destination`), move the pointer past every station within `current + tank` and remember the farthest one; stop there and count one stop. If no station lies ahead within reach, the trip is stuck, so print `-1`. Driving to the farthest reachable station is never worse, because any later stop that an earlier station could reach is also reachable from farther along. The pointer only moves forward, so after sorting the whole trip is one pass over at most `100000` stations.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
destination = int(tokens[0])
tank = int(tokens[1])
n = int(tokens[2])
stations = []
i = 0
while i < n:
    stations.append(int(tokens[3 + i]))
    i = i + 1
stations.sort()

stops = 0
current = 0
next_station = 0
stuck = False
while current + tank < destination and not stuck:
    farthest = current
    while next_station < n and stations[next_station] <= current + tank:
        farthest = stations[next_station]
        next_station = next_station + 1
    if farthest == current:
        stuck = True
    else:
        current = farthest
        stops = stops + 1
if stuck:
    print("-1")
else:
    print(str(stops))


## Question 5

**Notice:** keep both scores and `last_leader`, the most recent team that was ahead (`NONE` before anyone leads). After each event, work out the current leader. During a tie there is no leader, so `last_leader` stays unchanged, and a team that retakes the lead after a tie is not a change. When a team leads, count a change only if `last_leader` is the other team, then update `last_leader`. Each event is a constant amount of work, so `100000` events take one pass.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
score_a = 0
score_b = 0
last_leader = "NONE"
changes = 0
i = 0
while i < n:
    team = tokens[1 + 2 * i]
    points = int(tokens[2 + 2 * i])
    if team == "A":
        score_a = score_a + points
    else:
        score_b = score_b + points
    leader = "NONE"
    if score_a > score_b:
        leader = "A"
    elif score_b > score_a:
        leader = "B"
    if leader != "NONE":
        if last_leader != "NONE" and leader != last_leader:
            changes = changes + 1
        last_leader = leader
    i = i + 1
print(str(changes))


## Question 6

**Notice:** build one `(R+1)`-by-`(C+1)` 2D prefix-sum grid, then total each `K`-by-`K` placement with its top-left corner at `(top, left)` in four lookups: `prefix[bottom][right] - prefix[top][right] - prefix[bottom][left] + prefix[top][left]`, where `bottom = top + K` and `right = left + K`. Let the first placement set `best` so an all-negative grid still reports its true (negative) maximum. Building the grid and trying every placement each take about `R * C` steps, not `R * C * K * K`, so a million cells stay fast.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
rows = int(parts[0])
columns = int(parts[1])
size = int(parts[2])

prefix = []
row = 0
while row <= rows:
    prefix_row = []
    column = 0
    while column <= columns:
        prefix_row.append(0)
        column = column + 1
    prefix.append(prefix_row)
    row = row + 1

row = 0
while row < rows:
    column = 0
    while column < columns:
        value = int(parts[3 + row * columns + column])
        prefix[row + 1][column + 1] = (
            value
            + prefix[row][column + 1]
            + prefix[row + 1][column]
            - prefix[row][column]
        )
        column = column + 1
    row = row + 1

best = 0
top = 0
while top + size <= rows:
    left = 0
    while left + size <= columns:
        bottom = top + size
        right = left + size
        total = (
            prefix[bottom][right]
            - prefix[top][right]
            - prefix[bottom][left]
            + prefix[top][left]
        )
        if (top == 0 and left == 0) or total > best:
            best = total
        left = left + 1
    top = top + 1
print(str(best))
